# 04_test.ipynb — Final Test Set Evaluation under FREEZE Gate
This notebook evaluates the models on the official CIFAR test set:
1. **Auto-detects** attached output from `03_main` (Stage "final") containing `final_choice.json`.
2. **Sweep Reporting**: If `results_sweep.csv` is present, logs and reports sweep rows as validation numbers only.
3. **Copies** run folders into `/kaggle/working/runs/` treating input as read-only.
4. **DRY_RUN protection**: Default `DRY_RUN = True` lists what will be tested and stops before touching test data.
5. **FREEZE Protection**: Enforces `experiments/FREEZE.md`.
6. **Write-once**: `eval_test.py` writes `test_results.json` once per run and refuses to overwrite.


In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Setup working repository
REPO_SOURCE = "git"
REPO_URL = "https://github.com/PhoenixEvo/flower-lite.git"

if os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")

if not os.path.exists("eval_test.py"):
    if os.path.exists("flower-lite"):
        os.chdir("flower-lite")
        os.system("git pull")
    elif os.path.exists("/kaggle/working/flower-lite"):
        os.chdir("/kaggle/working/flower-lite")
        os.system("git pull")
    else:
        os.system(f"git clone {REPO_URL} flower-lite")
        os.chdir("flower-lite")

for p in [os.path.abspath("src"), "/kaggle/working/flower-lite/src"]:
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

os.system("pip install -e .")
import flowerlite

print("Working Directory:", os.getcwd())
print("FlowerLite package successfully imported from:", flowerlite.__file__)
assert os.path.exists("eval_test.py"), "eval_test.py must exist!"


In [ ]:
# 1. Auto-detect training output (final_choice.json from stage final)
search_patterns = [
    "/kaggle/input/*/final_choice.json",
    "/kaggle/input/*/*/final_choice.json",
]
candidates = []
for p in search_patterns:
    candidates.extend(glob.glob(p))

if not candidates and os.path.exists("experiments/final_choice.json"):
    candidates.append("experiments/final_choice.json")

print(f"Detected candidates for final_choice.json: {candidates}")
if len(candidates) == 0:
    raise FileNotFoundError(
        "Could not find any final_choice.json in /kaggle/input/*/ or /kaggle/input/*/*/. "
        "Please attach the output of 03_main (STAGE='final') as an input to this notebook!"
    )
elif len(candidates) > 1:
    raise ValueError(
        f"Ambiguous training output: found multiple final_choice.json files: {candidates}. "
        "Please ensure only one final stage output dataset is attached."
    )

choice_file = Path(candidates[0])
print(f"Selected final training output config: {choice_file}")
source_dir = choice_file.parent

with open(choice_file, "r") as f:
    final_choice = json.load(f)

print("Loaded final choices (selected strictly by validation):")
print(json.dumps(final_choice, indent=2))

# Check and report sweep results as validation numbers only
sweep_csv_candidates = glob.glob(str(source_dir / "*results_sweep.csv")) +                        glob.glob("/kaggle/input/*/results_sweep.csv") +                        glob.glob("/kaggle/input/*/*/results_sweep.csv")

if sweep_csv_candidates:
    import pandas as pd
    print(f"\n[SWEEP VALIDATION REPORT ONLY] Found sweep results: {sweep_csv_candidates[0]}")
    df_sw = pd.read_csv(sweep_csv_candidates[0])
    print(df_sw.to_string(index=False))

# 2. Treat input folder as read-only. Copy run folders to /kaggle/working/runs/
working_dir = Path("/kaggle/working")
working_runs_dir = working_dir / "runs"
working_runs_dir.mkdir(parents=True, exist_ok=True)

source_runs_dir = source_dir / "runs" if (source_dir / "runs").exists() else source_dir

print(f"\nCopying run folders from read-only {source_runs_dir} into {working_runs_dir}...")
copied_runs = []
for item in source_runs_dir.iterdir():
    if item.is_dir() and item.name != "runs":
        dest_run_dir = working_runs_dir / item.name
        dest_run_dir.mkdir(parents=True, exist_ok=True)
        for fname in ["best.pt", "last.pt", "history.csv", "FINISHED"]:
            src_f = item / fname
            if src_f.exists():
                shutil.copy2(src_f, dest_run_dir / fname)
        copied_runs.append(item.name)

print(f"Successfully copied {len(copied_runs)} runs into {working_runs_dir}: {copied_runs}")


In [ ]:
# 3. DRY-RUN Gate & FREEZE Marker
DRY_RUN = True  # Human: Set to False to perform actual test evaluation

print(f"DRY_RUN status: {DRY_RUN}")
if DRY_RUN:
    print("\n============================== DRY RUN MODE ==============================")
    print("The following chosen winner models (selected strictly by validation) WOULD be evaluated:")
    for dataset, info in final_choice.items():
        print(f"  * [{dataset.upper()}] Run: '{info['run_id']}' (Weights: '{info.get('weight_type', 'ema').upper()}')")
    print("\nThe following other finished runs WOULD be evaluated as 'reported, not used for selection':")
    for r_name in copied_runs:
        if not any(info.get("run_id") == r_name for info in final_choice.values()):
            d_name = "cifar10" if "c10" in r_name else "cifar100"
            print(f"  * [{d_name.upper()}] Run: '{r_name}'")
    print("\n[STOPPING] DRY_RUN is True. Test set has NOT been touched.")
    print("Please set DRY_RUN = False in this cell and re-run to perform final official evaluation.")
    print("==========================================================================")
    raise RuntimeError("DRY_RUN is True. Halted before touching test set. Set DRY_RUN = False to proceed.")

# Write FREEZE marker to unlock evaluation
os.makedirs("experiments", exist_ok=True)
freeze_file = Path("experiments/FREEZE.md")
with open(freeze_file, "w") as f:
    f.write("# FREEZE\nModel selection finalized. Test set unlocked for single-pass evaluation.\n")
print(f"Wrote {freeze_file}. Test set evaluation gate is now unlocked.")


In [ ]:
# 4. Evaluate chosen winner models on official test set
test_outputs = {}

for dataset, info in final_choice.items():
    run_id = info["run_id"]
    weight_type = info.get("weight_type", "ema")
    
    print(f"\nEvaluating WINNER for {dataset.upper()} (Run ID: {run_id}, Weights: {weight_type.upper()})")
    
    ckpt_dir = f"/kaggle/working/runs/{run_id}"
    if not os.path.exists(ckpt_dir):
        ckpt_dir = f"experiments/{run_id}"
        
    cmd = [
        sys.executable, "eval_test.py",
        "--run-id", run_id,
        "--dataset", dataset,
        "--weight-type", weight_type,
        "--checkpoint-dir", ckpt_dir,
        "--output-dir", ckpt_dir,
        "--data-root", "/kaggle/working/data"
    ]
    
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print("STDERR:\n", res.stderr)
        raise RuntimeError(f"eval_test.py failed for {run_id}")
        
    res_json_path = os.path.join(ckpt_dir, "test_results.json")
    with open(res_json_path, "r") as f:
        test_outputs[dataset] = json.load(f)


In [ ]:
# 5. Evaluate other finished runs ('reported, not used for selection') & print summary table
import pandas as pd

working_runs = Path("/kaggle/working/runs")
all_evaluations = []

for dataset, res in test_outputs.items():
    all_evaluations.append({
        "dataset": dataset,
        "run_id": res["run_id"],
        "weight_type": res["weight_type"],
        "top1": res["top1"],
        "top5": res["top5"],
        "macro_f1": res["macro_f1"],
        "nll": res["nll"],
        "ece": res["ece"],
        "role": "SELECTED WINNER (chosen by validation)"
    })

for run_dir in sorted(working_runs.iterdir()):
    if not run_dir.is_dir():
        continue
    run_id = run_dir.name
    if any(e["run_id"] == run_id for e in all_evaluations):
        continue
        
    dataset = "cifar10" if "c10" in run_id or "cifar10" in run_id else "cifar100"
    res_path = run_dir / "test_results.json"
    
    if not res_path.exists():
        cmd = [
            sys.executable, "eval_test.py",
            "--run-id", run_id,
            "--dataset", dataset,
            "--weight-type", "ema",
            "--checkpoint-dir", str(run_dir),
            "--output-dir", str(run_dir),
            "--data-root", "/kaggle/working/data"
        ]
        sub_res = subprocess.run(cmd, capture_output=True, text=True)
        if sub_res.returncode != 0:
            print(f"Skipping {run_id}: evaluation returned code {sub_res.returncode}")
            continue
            
    with open(res_path, "r") as f:
        metrics = json.load(f)
        all_evaluations.append({
            "dataset": dataset,
            "run_id": run_id,
            "weight_type": metrics.get("weight_type", "ema"),
            "top1": metrics.get("top1"),
            "top5": metrics.get("top5"),
            "macro_f1": metrics.get("macro_f1"),
            "nll": metrics.get("nll"),
            "ece": metrics.get("ece"),
            "role": "reported, not used for selection"
        })

df_all = pd.DataFrame(all_evaluations)
print("\n============================== FINAL OFFICIAL TEST RESULTS ==============================")
print(df_all.to_string(index=False))

with open("/kaggle/working/all_test_evaluations.json", "w") as f:
    json.dump(all_evaluations, f, indent=4)
print("\nWrote /kaggle/working/all_test_evaluations.json")
